# 📊 Notebook 01 — Exploratory Data Analysis (EDA)

**Khóa luận:** Phân tích Lan tỏa Rủi ro sử dụng NOTEARS Causal Discovery  
**Dữ liệu:** S&P500, VN-Index, Vàng, Dầu WTI, Bitcoin | 2018–2024  
**Chương tương ứng:** Chương 4.1 — Thống kê mô tả dữ liệu

---

## Mục tiêu
1. Thu thập và xử lý dữ liệu giá đóng cửa 5 thị trường
2. Tính log-return và rolling volatility 21 ngày
3. Kiểm định tính dừng (ADF / KPSS)
4. Thống kê mô tả: mean, std, skewness, kurtosis, Jarque-Bera
5. Vẽ correlation heatmap và time series volatility

## 0. Setup

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')   # KLTN root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from src.utils.logger import setup_logger
setup_logger('KLTN')

# Matplotlib style
plt.rcParams.update({
    'figure.dpi'     : 120,
    'font.family'    : 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('✅ Setup hoàn tất')

✅ Setup hoàn tất


---
## 1. Thu thập & xử lý dữ liệu

In [2]:
from src.data.collector    import DataCollector
from src.data.preprocessor import DataPreprocessor
from config import NODE_NAMES, MARKET_LABELS

# Thu thập dữ liệu
collector = DataCollector(vnindex_path='../data/raw/vnindex_manual.csv')
prices    = collector.fetch()

print(f'Raw prices shape: {prices.shape}')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')
prices.tail(3)

ModuleNotFoundError: No module named 'src.data.collector'

In [ ]:
# Xử lý: log-return + rolling volatility
prep   = DataPreprocessor()
X, vol = prep.fit_transform(prices, standardize=True)

vol_df = prep.volatility_   # volatility chưa chuẩn hóa (để plot)
print(f'Volatility shape : {vol_df.shape}')
print(f'X (normalized)   : {X.shape}')
vol_df.head(3)

---
## 2. Thống kê mô tả (Bảng 4.1)

In [ ]:
desc = prep.describe()
print('\n📋 Bảng thống kê mô tả — Rolling Volatility 21 ngày\n')
desc.style \
    .background_gradient(cmap='YlOrRd', subset=['Std','Skewness','Kurtosis']) \
    .format(precision=4)

In [ ]:
# Nhận xét tự động
print('📌 Nhận xét:')
for market in desc.index:
    row = desc.loc[market]
    kurt = row['Kurtosis']
    skew = row['Skewness']
    jb_p = row['JB p-value']
    label = MARKET_LABELS.get(market, market)
    fat_tail = 'fat-tail' if kurt > 3 else 'normal tail'
    dist     = 'KHÔNG chuẩn' if jb_p < 0.05 else 'chuẩn'
    print(f'  {label:<15s}: skew={skew:+.3f} | kurt={kurt:.3f} ({fat_tail}) | JB p={jb_p:.4f} → phân phối {dist}')

---
## 3. Kiểm định tính dừng ADF / KPSS (Bảng 4.2)

In [ ]:
from src.data.validator import StationarityValidator

validator  = StationarityValidator(alpha=0.05)
stat_table = validator.test(vol_df)

print('\n📋 Kết quả kiểm định tính dừng\n')
stat_table.style \
    .applymap(
        lambda v: 'background-color: #c8f7c5' if 'Dừng' in str(v)
                  else ('background-color: #f7c8c8' if 'Phi' in str(v) else ''),
        subset=['Kết luận tổng hợp']
    )

---
## 4. Visualizations

In [ ]:
# 4.1 Rolling Volatility Time Series
from src.visualization.time_series import plot_volatility_series

fig = plot_volatility_series(
    vol_df      = vol_df,
    mark_events = True,
    output_path = '../reports/figures/01_volatility_series.png',
)
plt.show()

In [ ]:
# 4.2 Correlation Heatmap
from src.visualization.time_series import plot_correlation_heatmap

fig = plot_correlation_heatmap(
    vol_df      = vol_df,
    output_path = '../reports/figures/01_correlation_heatmap.png',
)
plt.show()

print('\n📌 Nhận xét tương quan:')
corr = vol_df.corr()
for i in range(len(NODE_NAMES)):
    for j in range(i+1, len(NODE_NAMES)):
        c = corr.iloc[i, j]
        m1 = MARKET_LABELS.get(NODE_NAMES[i], NODE_NAMES[i])
        m2 = MARKET_LABELS.get(NODE_NAMES[j], NODE_NAMES[j])
        level = 'cao' if abs(c) > 0.5 else ('trung bình' if abs(c) > 0.3 else 'thấp')
        print(f'  {m1} ↔ {m2}: r={c:.3f} ({level})')

In [ ]:
# 4.3 Descriptive Stats Bar Charts
from src.visualization.time_series import plot_descriptive_stats

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ['Std', 'Kurtosis']):
    plot_descriptive_stats(desc, metric=metric)
plt.tight_layout()
plt.savefig('../reports/figures/01_descriptive_stats.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Phân tích giai đoạn đặc biệt (COVID / Crypto Crash)

In [ ]:
events = {
    'Pre-COVID'   : ('2018-01-01', '2020-02-28'),
    'COVID-19'    : ('2020-03-01', '2021-06-30'),
    'Crypto-Crash': ('2022-05-01', '2022-12-31'),
}

print('📊 Volatility trung bình theo giai đoạn:\n')
rows = []
for period, (s, e) in events.items():
    sub = vol_df.loc[s:e]
    row = {'Giai đoạn': period}
    for col in vol_df.columns:
        row[MARKET_LABELS.get(col, col)] = round(sub[col].mean(), 5)
    rows.append(row)

period_df = pd.DataFrame(rows).set_index('Giai đoạn')
period_df.style.background_gradient(cmap='YlOrRd', axis=None).format('{:.5f}')

---
## 6. Lưu kết quả

In [ ]:
import os
os.makedirs('../reports/tables', exist_ok=True)

desc.to_csv('../reports/tables/01_descriptive_stats.csv')
stat_table.to_csv('../reports/tables/01_stationarity_tests.csv')
period_df.to_csv('../reports/tables/01_period_volatility.csv')

# Lưu vol_df để các notebook khác dùng
vol_df.to_csv('../reports/tables/volatility.csv')

print('✅ Đã lưu tất cả kết quả EDA vào reports/')